In [12]:
import numpy as np
import xml.etree.ElementTree as ET
from typing import List, Tuple, Dict
from xml.dom import minidom

def parse_polylines_by_image(xml_content: str) -> Dict[str, Dict]:
    """Parse polyline points from XML content, grouped by image name."""
    root = ET.fromstring(xml_content)
    image_polylines = {}
    
    for image in root.findall('.//image'):
        image_name = image.get('name')
        width = int(image.get('width'))
        height = int(image.get('height'))
        polylines = []
        
        for polyline in image.findall('.//polyline'):
            points_str = polyline.get('points')
            points = [tuple(map(float, point.split(','))) 
                     for point in points_str.split(';')]
            polylines.append(points)
        
        if polylines:
            image_polylines[image_name] = {
                'polylines': polylines,
                'width': width,
                'height': height
            }
    
    return image_polylines

def normalize_vector(v: np.ndarray) -> np.ndarray:
    """Normalize a vector."""
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm

def polyline_to_polygon(points: List[Tuple[float, float]], width: float = 1.0) -> List[Tuple[float, float]]:
    """Convert a polyline to a polygon by giving it width."""
    if len(points) < 2:
        return []
    
    # Convert points to numpy array for easier calculation
    points = np.array(points)
    
    # Calculate vectors between consecutive points
    vectors = points[1:] - points[:-1]
    
    # Calculate normalized perpendicular vectors
    perp_vectors = np.zeros_like(vectors)
    perp_vectors[:, 0] = -vectors[:, 1]
    perp_vectors[:, 1] = vectors[:, 0]
    perp_vectors = np.array([normalize_vector(v) for v in perp_vectors])
    
    # Calculate the offset for each segment
    half_width = width / 2
    offsets = perp_vectors * half_width
    
    # Create the polygon points
    upper_points = []
    lower_points = []
    
    # Handle first point
    upper_points.append(tuple(points[0] + offsets[0]))
    lower_points.append(tuple(points[0] - offsets[0]))
    
    # Handle middle points
    for i in range(1, len(points) - 1):
        # Average the offset vectors for smooth transitions
        avg_offset = normalize_vector(offsets[i-1] + offsets[i]) * half_width
        upper_points.append(tuple(points[i] + avg_offset))
        lower_points.append(tuple(points[i] - avg_offset))
    
    # Handle last point
    upper_points.append(tuple(points[-1] + offsets[-1]))
    lower_points.append(tuple(points[-1] - offsets[-1]))
    
    # Combine points to form polygon (go up one side and down the other)
    polygon_points = upper_points + lower_points[::-1]
    
    return polygon_points

def create_xml_output(image_polygons: Dict[str, List[List[Tuple[float, float]]]]) -> str:
    """Create XML output with polygon annotations."""
    root = ET.Element('annotations')
    
    for image_name, data in image_polygons.items():
        image_elem = ET.SubElement(root, 'image')
        image_elem.set('name', image_name)
        image_elem.set('width', str(data['width']))
        image_elem.set('height', str(data['height']))
        
        for polygon_points in data['polygons']:
            polygon = ET.SubElement(image_elem, 'polygon')
            polygon.set('label', 'breaker')
            polygon.set('source', 'auto')
            polygon.set('occluded', '0')
            
            # Convert points to string format
            points_str = ';'.join(f'{x:.2f},{y:.2f}' for x, y in polygon_points)
            polygon.set('points', points_str)
            
            polygon.set('z_order', '0')
    
    # Pretty print the XML
    xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent='  ')
    return xml_str

def main(xml_content: str, line_width: float = 1.0) -> str:
    """Main function to process XML and create polygon annotations."""
    # Parse original polylines
    image_data = parse_polylines_by_image(xml_content)
    
    # Process each image
    result_data = {}
    for image_name, data in image_data.items():
        # Convert each polyline to a polygon
        polygons = []
        for polyline in data['polylines']:
            polygon = polyline_to_polygon(polyline, width=line_width)
            if polygon:
                polygons.append(polygon)
        
        # Store results
        result_data[image_name] = {
            'polygons': polygons,
            'width': data['width'],
            'height': data['height']
        }
        
        print(f"Processed {image_name}:")
        print(f"  Original polylines: {len(data['polylines'])}")
        print(f"  Generated polygons: {len(polygons)}")
    
    # Generate XML output
    return create_xml_output(result_data)

# Read input XML
with open('cvat_test/annotations.xml', 'r') as f:
    xml_content = f.read()

# Process and generate new XML with thin polygons
output_xml = main(xml_content, line_width=1.0)

# Save output XML
with open('cvat_test/annotations_poly.xml', 'w') as f:
    f.write(output_xml)

print("\nCreated annotations_poly.xml")

Processed 20211014T120100Z_t1553-2596_a164.png:
  Original polylines: 31
  Generated polygons: 31
Processed 20211014T120100Z_t1553-2596_a165.png:
  Original polylines: 33
  Generated polygons: 33
Processed 20211014T120100Z_t1553-2596_a166.png:
  Original polylines: 32
  Generated polygons: 32
Processed 20211014T120100Z_t1553-2596_a203.png:
  Original polylines: 34
  Generated polygons: 34
Processed 20211014T120100Z_t1553-2596_a204.png:
  Original polylines: 25
  Generated polygons: 25
Processed 20211014T120100Z_t1553-2596_a205.png:
  Original polylines: 27
  Generated polygons: 27
Processed 20211014T120100Z_t1553-2596_a386.png:
  Original polylines: 34
  Generated polygons: 34
Processed 20211014T120100Z_t1553-2596_a387.png:
  Original polylines: 32
  Generated polygons: 32
Processed 20211014T120100Z_t1553-2596_a388.png:
  Original polylines: 31
  Generated polygons: 31
Processed 20211014T120100Z_t1553-2596_a389.png:
  Original polylines: 22
  Generated polygons: 22
Processed 20211014T1

In [13]:
import numpy as np
import cv2
import xml.etree.ElementTree as ET
from typing import List, Tuple, Dict
import distinctipy

def parse_polygons_from_xml(xml_file: str) -> Dict[str, Dict]:
    """Parse polygon points from XML file."""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    image_polygons = {}
    
    for image in root.findall('.//image'):
        image_name = image.get('name')
        width = int(image.get('width'))
        height = int(image.get('height'))
        
        polygons = []
        for polygon in image.findall('.//polygon'):
            points_str = polygon.get('points')
            points = [tuple(map(float, point.split(','))) 
                     for point in points_str.split(';')]
            polygons.append(points)
            
        if polygons:
            image_polygons[image_name] = {
                'polygons': polygons,
                'width': width,
                'height': height
            }
    
    return image_polygons

def create_colored_mask(polygons: List[List[Tuple[float, float]]], 
                       width: int, 
                       height: int) -> np.ndarray:
    """Create a colored mask where each polygon has a distinct color."""
    # Create empty RGB mask
    mask = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Get distinct colors for each polygon
    n_colors = len(polygons)
    colors = distinctipy.get_colors(n_colors)
    
    # Convert colors to BGR (OpenCV format) and scale to 0-255
    colors_bgr = [(int(b * 255), int(g * 255), int(r * 255)) 
                  for r, g, b in colors]
    
    # Draw each polygon with its distinct color
    for polygon, color in zip(polygons, colors_bgr):
        # Convert points to integer array
        points = np.array(polygon, dtype=np.int32)
        
        # Draw filled polygon
        cv2.fillPoly(mask, [points], color)
    
    return mask

def main(xml_file: str, output_dir: str = 'visualization') -> None:
    """Main function to create colored visualizations of polygons."""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Parse polygons from XML
    image_data = parse_polygons_from_xml(xml_file)
    
    print(f"Found {len(image_data)} images with polygons")
    
    # Process each image
    for image_name, data in image_data.items():
        # Create colored mask
        mask = create_colored_mask(
            data['polygons'], 
            data['width'], 
            data['height']
        )
        
        # Save visualization
        output_name = f"viz_{image_name}"
        output_path = os.path.join(output_dir, output_name)
        cv2.imwrite(output_path, mask)
        
        print(f"\nProcessed {image_name}:")
        print(f"  Number of polygons: {len(data['polygons'])}")
        print(f"  Output saved as: {output_name}")

# Generate visualizations
main('cvat_test/annotations_poly.xml', 'cvat_test/visualization')

Found 64 images with polygons

Processed 20211014T120100Z_t1553-2596_a164.png:
  Number of polygons: 31
  Output saved as: viz_20211014T120100Z_t1553-2596_a164.png

Processed 20211014T120100Z_t1553-2596_a165.png:
  Number of polygons: 33
  Output saved as: viz_20211014T120100Z_t1553-2596_a165.png

Processed 20211014T120100Z_t1553-2596_a166.png:
  Number of polygons: 32
  Output saved as: viz_20211014T120100Z_t1553-2596_a166.png

Processed 20211014T120100Z_t1553-2596_a203.png:
  Number of polygons: 34
  Output saved as: viz_20211014T120100Z_t1553-2596_a203.png

Processed 20211014T120100Z_t1553-2596_a204.png:
  Number of polygons: 25
  Output saved as: viz_20211014T120100Z_t1553-2596_a204.png

Processed 20211014T120100Z_t1553-2596_a205.png:
  Number of polygons: 27
  Output saved as: viz_20211014T120100Z_t1553-2596_a205.png

Processed 20211014T120100Z_t1553-2596_a386.png:
  Number of polygons: 34
  Output saved as: viz_20211014T120100Z_t1553-2596_a386.png

Processed 20211014T120100Z_t155